# Stage 1 — The portfolio you built

**Checkpoint:** `stage-1`  ·  run `python verify.py` first — it should say *Stage 1 verified*.

You generated 12,000 small businesses that never existed. This notebook is the visual version
of the audit on your lab sheet: are the counts right, is anything null, and does the data
**rank-order** the way credit intuition says it should?

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "shared/data/raw/businesses.parquet",
    "shared/data/raw/portfolio.parquet",
    "shared/data/raw/panel.parquet",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-1 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-1      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-1 artifacts present.")

## 1. Row counts — do they match the lab sheet exactly?

In [ ]:
from shared.config import RAW

businesses = pd.read_parquet(RAW / "businesses.parquet")
portfolio  = pd.read_parquet(RAW / "portfolio.parquet")
panel      = pd.read_parquet(RAW / "panel.parquet")

counts = pd.DataFrame(
    [{"file": "businesses.parquet", "rows": len(businesses), "columns": businesses.shape[1]},
     {"file": "portfolio.parquet",  "rows": len(portfolio),  "columns": portfolio.shape[1]},
     {"file": "panel.parquet",      "rows": len(panel),      "columns": panel.shape[1]}]
).set_index("file")

expected = {"businesses.parquet": 12000, "portfolio.parquet": 8336, "panel.parquet": 200064}
counts["expected"] = pd.Series(expected)
counts["match"] = np.where(counts["rows"] == counts["expected"], "OK", "MISMATCH")
counts

> **Why is 8,336 not a round number?** It is the *booked* subset — the applicants who were
> approved **and** took the money. It falls out of the process rather than being chosen, which
> is exactly how a real book accumulates.

## 2. Target rates

In [ ]:
rates = pd.Series({
    "default rate (applicants)":        businesses["default"].mean(),
    "booked rate (applicants)":         businesses["booked"].mean(),
    "deterioration rate (on-book)":     portfolio["deterioration_next_6_12mo"].mean(),
    "line_increase_good rate (on-book)":portfolio["line_increase_good"].mean(),
}).round(4)
rates.to_frame("rate")

## 3. Null audit — synthetic data is complete by construction, so a null is a bug

In [ ]:
nulls = pd.Series({
    "businesses": int(businesses.isna().sum().sum()),
    "portfolio":  int(portfolio.isna().sum().sum()),
    "panel":      int(panel.isna().sum().sum()),
}, name="null cells")
print(nulls.to_string())
print("\nPASS — no nulls anywhere." if nulls.sum() == 0 else "\nFAIL — investigate.")

## 4. Default rate by industry — does any segment look implausible?

In [ ]:
by_ind = (businesses.groupby("industry")["default"]
          .agg(["size", "mean"]).sort_values("mean", ascending=False))
by_ind.columns = ["applicants", "default_rate"]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(by_ind.index, by_ind["default_rate"] * 100, color="#5b8ac6")
ax.axvline(businesses["default"].mean() * 100, color="#b3372e", ls="--", lw=1.5,
           label=f"portfolio average {businesses['default'].mean():.1%}")
ax.invert_yaxis(); ax.set_xlabel("default rate (%)"); ax.legend()
ax.set_title("Default rate by industry")
plt.tight_layout(); plt.show()

by_ind.style.format({"default_rate": "{:.1%}"})

## 5. Rank-ordering — the sanity check that matters

Credit intuition says: **higher DSCR → safer**, **higher utilization → riskier.** If the data
disagrees, either the intuition or the generator is wrong. This is the check a validator runs
before looking at any model.

In [ ]:
def decile_rate(df, col, target="default", q=10, ascending_expect="down"):
    d = df[[col, target]].copy()
    d["bucket"] = pd.qcut(d[col], q=q, duplicates="drop")
    out = d.groupby("bucket", observed=True)[target].agg(["size", "mean"])
    out.columns = ["n", "default_rate"]
    return out

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (col, expect) in zip(axes, [("dscr", "falls"), ("utilization", "rises")]):
    t = decile_rate(businesses, col)
    ax.plot(range(len(t)), t["default_rate"] * 100, marker="o", color="#245bb2")
    ax.set_title(f"{col} deciles — default rate should {expect}")
    ax.set_xlabel(f"{col} decile (low → high)"); ax.set_ylabel("default rate (%)")
plt.tight_layout(); plt.show()

dscr_t = decile_rate(businesses, "dscr")
util_t = decile_rate(businesses, "utilization")
print(f"dscr:        first decile {dscr_t['default_rate'].iloc[0]:.1%} -> last {dscr_t['default_rate'].iloc[-1]:.1%}  ({'OK, falls' if dscr_t['default_rate'].iloc[0] > dscr_t['default_rate'].iloc[-1] else 'UNEXPECTED'})")
print(f"utilization: first decile {util_t['default_rate'].iloc[0]:.1%} -> last {util_t['default_rate'].iloc[-1]:.1%}  ({'OK, rises' if util_t['default_rate'].iloc[-1] > util_t['default_rate'].iloc[0] else 'UNEXPECTED'})")

## 6. The behavioural panel — 24 months of it feeds Session 4

In [ ]:
monthly = panel.groupby("month_index").agg(
    utilization=("utilization", "mean"),
    days_past_due=("days_past_due", "mean"),
    deposit_inflow=("deposit_inflow", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, title in zip(axes, ["utilization", "days_past_due", "deposit_inflow"],
                          ["mean utilization", "mean days past due", "mean deposit inflow"]):
    ax.plot(monthly["month_index"], monthly[col], color="#245bb2")
    ax.set_title(title); ax.set_xlabel("month")
plt.tight_layout(); plt.show()

print(f"{panel['business_id'].nunique():,} businesses x {panel['month_index'].nunique()} months = {len(panel):,} rows")

## 7. The deny-list — six columns no model may ever see

A deny-list in someone's head is not a control. A deny-list in `config.py` is.

In [ ]:
from shared.config import LEAKAGE_COLUMNS

why = {
    "pd_default_origination":     "the data generator's ground-truth PD — the answer, laundered",
    "default":                    "the answer itself",
    "risk_based_rate":            "priced FROM the true PD, so it leaks it",
    "booked":                     "a post-decision outcome; unknown at decision time",
    "deterioration_next_6_12mo":  "Session 4's target — future information",
    "line_increase_good":         "Session 4's other target — future information",
}
pd.DataFrame({"column": LEAKAGE_COLUMNS,
              "present in businesses?": [c in businesses.columns for c in LEAKAGE_COLUMNS],
              "why it is forbidden": [why.get(c, "") for c in LEAKAGE_COLUMNS]}).set_index("column")

### Feel the poison (optional, 20 seconds)

Train on a denied column and watch the model become suspiciously perfect. In production this
is what a silent join mistake looks like — and it will not announce itself with a column
helpfully named `cheat`.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X_ok  = businesses[["dscr", "leverage", "utilization", "prior_delinquencies"]]
X_bad = X_ok.assign(cheat=businesses["pd_default_origination"])   # deny-listed
y = businesses["default"]

for name, X in [("honest", X_ok), ("LEAKED", X_bad)]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
    m = LGBMClassifier(verbose=-1).fit(Xtr, ytr)
    print(f"{name:>7}: AUC = {roc_auc_score(yte, m.predict_proba(Xte)[:, 1]):.4f}")

---
**Next:** `02_stage2_score_spine.ipynb` — turn this portfolio into a credit score that has to
pass a hard gate, or it does not ship.